# 🎬 Wan 2.2 (Alibaba Open Video) + ComfyUI on Google Colab

สมุดงาน Colab สำหรับรัน **Wan 2.2** ร่วมกับ **ComfyUI**
ดึงโมเดลตรงจาก **Hugging Face (`Comfy-Org/Wan-2.2`)** สำหรับเชื่อมต่อกับระบบ **PrintFilm Web App**

### 🌟 จุดเด่น
- **Wan 2.2 TI2V (5B):** สถาปัตยกรรม Text-and-Image to Video ล็อกภาพ **Cover Shot** เป็น First Frame ตรงปก ไม่หลอนพูดเอง
- **เสถียรและเร็ว:** มี `--enable-cors-header` และ `--enable-compress-response-body` ป้องกันปัญหา Cloudflare Error 1033 หรือหน้าเว็บค้าง
- **เปิดได้ 2 ช่องทาง:**
  1. **Colab Proxy ในตัว:** กดเปิดหน้า ComfyUI ได้ทันทีในแท็บใหม่
  2. **Cloudflare Tunnel:** ได้ลิงก์ `https://....trycloudflare.com` สำหรับนำไปใส่ในเว็บ PrintFilm

---
### 🚀 ขั้นตอนการใช้งาน
1. รันเซลล์ **① ตรวจสอบ GPU และติดตั้ง ComfyUI** (~1 นาที)
2. รันเซลล์ **② ดาวน์โหลดโมเดล Wan 2.2 จาก Hugging Face** (~2 นาที ด้วย aria2)
3. รันเซลล์ **③ สตาร์ท ComfyUI**
4. รันเซลล์ **④ เปิด Cloudflare Tunnel** แล้วคัดลอกลิงก์ไปใส่ในเว็บ PrintFilm

In [ ]:
#@title ① ตรวจสอบสภาพแวดล้อม GPU + ติดตั้ง ComfyUI และ Custom Nodes
import torch, shutil, subprocess, sys, os
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError('❌ ไม่พบ GPU: กรุณาไปที่ Runtime -> Change runtime type แล้วเลือก L4 GPU หรือ A100 (หรือ T4)')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
print(f'✅ ตรวจพบ GPU: {gpu_name} (VRAM {vram_gb} GB)')

COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    print('🚀 กำลังติดตั้ง ComfyUI (เวอร์ชันล่าสุดที่รองรับ Wan 2.2)...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY)], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
    subprocess.run(['pip', 'install', '-q', 'accelerate', 'diffusers', 'torchsde', 'einops', 'av'], check=True)

# ติดตั้ง Custom Nodes
CUSTOM = COMFY / 'custom_nodes'
manager = CUSTOM / 'ComfyUI-Manager'
if not manager.exists():
    print('📦 ติดตั้ง ComfyUI-Manager...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ltdrdata/ComfyUI-Manager.git', str(manager)], check=True)

vhs = CUSTOM / 'ComfyUI-VideoHelperSuite'
if not vhs.exists():
    print('📦 ติดตั้ง VideoHelperSuite (สำหรับ Export MP4 H264)...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git', str(vhs)], check=True)
    subprocess.run(['pip', 'install', '-q', '-r', str(vhs / 'requirements.txt')], check=True)

fmlf = CUSTOM / 'ComfyUI-Wan22FMLF'
if not fmlf.exists():
    print('📦 ติดตั้ง ComfyUI-Wan22FMLF (ควบคุม First-Middle-Last Frame และล็อกใบหน้าตรงปก)...')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/wallen0322/ComfyUI-Wan22FMLF.git', str(fmlf)], check=True)
    if (fmlf / 'requirements.txt').exists():
        subprocess.run(['pip', 'install', '-q', '-r', str(fmlf / 'requirements.txt')], check=True)

print('✅ ติดตั้ง ComfyUI และ Custom Nodes เรียบร้อยพร้อมใช้งาน!')


In [ ]:
#@title ② ดาวน์โหลดโมเดล Wan 2.2 จาก Hugging Face (Comfy-Org/Wan-2.2)
import subprocess, os
from pathlib import Path

# เลือกโมเดลที่ต้องการ:
# 'Wan2.2-TI2V-5B' (แนะนำที่สุด: เบา รวดเร็ว โหลดเพียง ~9GB ไม่หลอนพูดเอง ล็อก First Frame)
# 'Bernini-R (Wan2.2-14B-FP8)' (ByteDance Bernini-R แม่นยำเรื่อง Reference ภาพตัวละครและฉากที่สุด)
# 'Wan2.2-14B-Dual' (รุ่นใหญ่ 14B High+Low Noise + 4-Step LoRA รวม ~28GB)
MODEL_VERSION = 'Wan2.2-TI2V-5B'  #@param ['Wan2.2-TI2V-5B', 'Bernini-R (Wan2.2-14B-FP8)', 'Wan2.2-14B-Dual']

subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], check=True)

COMFY = Path('/content/ComfyUI')
diff_folder = COMFY / 'models' / 'diffusion_models'
text_folder = COMFY / 'models' / 'text_encoders'
vae_folder = COMFY / 'models' / 'vae'
lora_folder = COMFY / 'models' / 'loras'

for f in [diff_folder, text_folder, vae_folder, lora_folder]:
    f.mkdir(parents=True, exist_ok=True)

def aria_download(url: str, out_dir: Path, out_name: str):
    target = out_dir / out_name
    if target.exists() and target.stat().st_size > 1024 * 1024:
        print(f'⏩ มีไฟล์แล้ว: {out_name} ({target.stat().st_size / (1024**3):.2f} GB)')
        return
    print(f'⬇️ กำลังโหลด: {out_name}...')
    cmd = [
        'aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M',
        url, '-d', str(out_dir), '-o', out_name
    ]
    subprocess.run(cmd, check=True)
    print(f'✅ สำเร็จ: {out_name}')

if MODEL_VERSION == 'Wan2.2-TI2V-5B':
    # 1. Diffusion Model Wan 2.2 TI2V 5B (~9.31 GB)
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',
        diff_folder,
        'wan2.2_ti2v_5B_fp16.safetensors'
    )
    # 2. VAE Wan 2.2 (~1.31 GB)
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/vae/wan2.2_vae.safetensors',
        vae_folder,
        'wan2.2_vae.safetensors'
    )
elif MODEL_VERSION == 'Bernini-R (Wan2.2-14B-FP8)':
    # ByteDance Bernini-R High + Low Noise Models (Comfy-Org/Bernini-R)
    aria_download(
        'https://huggingface.co/Comfy-Org/Bernini-R/resolve/main/diffusion_models/wan2.2_bernini_r_high_noise_fp8_scaled.safetensors',
        diff_folder,
        'wan2.2_bernini_r_high_noise_fp8_scaled.safetensors'
    )
    aria_download(
        'https://huggingface.co/Comfy-Org/Bernini-R/resolve/main/diffusion_models/wan2.2_bernini_r_low_noise_fp8_scaled.safetensors',
        diff_folder,
        'wan2.2_bernini_r_low_noise_fp8_scaled.safetensors'
    )
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/vae/wan2.2_vae.safetensors',
        vae_folder,
        'wan2.2_vae.safetensors'
    )
else:
    # Wan 2.2 14B High + Low Noise Models
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors',
        diff_folder,
        'wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors'
    )
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors',
        diff_folder,
        'wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors'
    )
    # LoRAs (4-Step)
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors',
        lora_folder,
        'wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors'
    )
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors',
        lora_folder,
        'wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors'
    )
    aria_download(
        'https://huggingface.co/Comfy-Org/Wan-2.2/resolve/main/split_files/vae/wan2.2_vae.safetensors',
        vae_folder,
        'wan2.2_vae.safetensors'
    )

# 3. Shared Text Encoder UMT5 XXL FP8 (~6.27 GB)
aria_download(
    'https://huggingface.co/Comfy-Org/Wan-2.1/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    text_folder,
    'umt5_xxl_fp8_e4m3fn_scaled.safetensors'
)

print('\n🎉 ดาวน์โหลดโมเดล Wan 2.2 ครบถ้วนแล้ว!')


In [ ]:
#@title ③ สตาร์ท ComfyUI (เปิด CORS Header + บีบอัด Response Body)
import subprocess, time, urllib.request, sys, os
from pathlib import Path

COMFY = Path('/content/ComfyUI')

# ปิด instance เดิมถ้ามีค้างอยู่
if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    server_proc.wait()
    print('ปิด instance เก่าแล้ว กำลังรีสตาร์ท...')

# สร้าง config manager
cfg = COMFY / 'user' / '__manager' / 'config.ini'
cfg.parent.mkdir(parents=True, exist_ok=True)
if not cfg.exists():
    cfg.write_text('[default]\nsecurity_level = normal-\n', encoding='utf-8')

log_file = open('/content/comfyui.log', 'w')
cmd = [
    sys.executable, 'main.py',
    '--listen', '127.0.0.1',
    '--port', '8188',
    '--enable-cors-header',
    '--enable-compress-response-body',
    '--fp8_e4m3fn-text-enc'
]

print('🚀 กำลังสตาร์ท ComfyUI พร้อมตั้งค่า CORS และ Response Compression...')
server_proc = subprocess.Popen(cmd, cwd=str(COMFY), stdout=log_file, stderr=subprocess.STDOUT)

# รอจนกระทั่งระบบพร้อม 100%
started = False
for i in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        started = True
        print('✅ ComfyUI สตาร์ทสำเร็จและพร้อมทำงานแล้ว!')
        break
    except Exception:
        time.sleep(2)

if not started:
    print('❌ ComfyUI สตาร์ทไม่สำเร็จ ดู 50 บรรทัดล่าสุดของ log:')
    if os.path.exists('/content/comfyui.log'):
        print(''.join(open('/content/comfyui.log').readlines()[-50:]))
    raise RuntimeError('ComfyUI startup timeout')

# ตัวเลือกที่ 1: เปิดดูหน้า UI ใน Colab ได้ทันที (เร็ว เสถียร ไม่ผ่าน Cloudflare)
try:
    from google.colab import output
    print('👉 คลิกเปิดหน้าต่าง ComfyUI ตรงๆ ใน Colab:')
    output.serve_kernel_port_as_window(8188)
except Exception as e:
    pass


In [ ]:
#@title ④ เปิด Tunnel เชื่อมต่อภายนอก (นำ URL ไปใส่ในเว็บ PrintFilm)
TUNNEL_SERVICE = 'Cloudflare'  #@param ['Cloudflare', 'Ngrok']
NGROK_AUTHTOKEN = ''  #@param {type:'string'}

import subprocess, time, re, os
from pathlib import Path

tunnel_url = None
if TUNNEL_SERVICE == 'Ngrok':
    if not NGROK_AUTHTOKEN.strip():
        raise ValueError('กรุณากรอก NGROK_AUTHTOKEN (สมัครฟรีได้ที่ https://dashboard.ngrok.com/get-started/your-authtoken)')
    subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    ngrok.kill()
    tunnel = ngrok.connect(8188)
    tunnel_url = tunnel.public_url
else:
    # ใช้ Cloudflare Tunnel
    cf_bin = Path('/content/cloudflared')
    if not cf_bin.exists():
        print('⬇️ ดาวน์โหลด Cloudflare Tunnel Client...')
        subprocess.run(['wget', '-q', '-O', str(cf_bin), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
        subprocess.run(['chmod', '+x', str(cf_bin)], check=True)

    if 'cf_proc' in globals() and cf_proc.poll() is None:
        cf_proc.terminate()
        cf_proc.wait()

    open('/content/cloudflared.log', 'w').close()
    cf_log = open('/content/cloudflared.log', 'a')
    cf_proc = subprocess.Popen([
        str(cf_bin), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'
    ], stdout=cf_log, stderr=subprocess.STDOUT)

    for _ in range(60):
        time.sleep(1)
        if os.path.exists('/content/cloudflared.log'):
            cf_txt = open('/content/cloudflared.log', encoding='utf-8', errors='ignore').read()
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', cf_txt)
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print('\n===================================================================')
    print(f'🎯 URL สำหรับนำไปใส่ในเว็บ PrintFilm:\n   {tunnel_url}')
    print('===================================================================')
    print('📋 นำ URL นี้ไปใส่ในช่อง "ComfyUI URL" บนเว็บ PrintFilm ได้เลย!')
else:
    print('⚠️ ยังไม่ได้ URL กรุณาตรวจดู /content/cloudflared.log')


In [ ]:
#@title 🔍 ตรวจสอบสถานะ GPU และ ComfyUI Log (กรณีเกิดปัญหา)
import subprocess, os
print('--- NVIDIA-SMI ---')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

print('\n--- COMFYUI LOG (100 บรรทัดล่าสุด) ---')
if os.path.exists('/content/comfyui.log'):
    lines = open('/content/comfyui.log', encoding='utf-8', errors='ignore').readlines()
    print(''.join(lines[-100:]))
